# CMPE492 — Extended Metrics (FID, FVD, LPIPS, PSNR, SSIM)

**Purpose:** Compute additional visual-quality and temporal-coherence metrics on the multi-identity benchmark, complementing the existing SyncNet (lip-sync) and ArcFace (identity) metrics.

**Metric families:**
- **Distribution-based (vs real HDTF videos — ideal for cross-audio):**
  - **FID** ↓ — Fréchet Inception Distance: how realistic generated frames look vs real frames
  - **FVD** ↓ — Fréchet Video Distance: temporal realism of generated clips vs real clips
- **Reference-based (vs raw HDTF video, frame-aligned):**
  - **LPIPS** ↓ — perceptual distance per frame
  - **PSNR** ↑ — pixel-level fidelity
  - **SSIM** ↑ — structural similarity

> ⚠️ **Cross-audio caveat:** Since the generated videos use cross-identity audio, the *lip region* differs from the raw video by design. So LPIPS/PSNR/SSIM-vs-raw primarily reflect identity, background, and pose fidelity (most of the frame) and will be penalized somewhat by the intended lip differences. FID and FVD (distribution-based) are the cleaner headline metrics for this setting.

**Inputs:**
- `batch_eval.zip` (Stage 3 output: hybrid_lp/, lp_only/, references/)
- `hdtf_prepared.zip` (Stage 1: raw/ videos = real reference)

---
**Run order:** 1 → 2 → 3 → 4 → 5 → 6 → 7

## Cell 1 — Setup

In [ ]:
import subprocess, sys

pkgs = ['torchmetrics', 'lpips', 'scikit-image', 'pytorch-fid', 'cd-fvd']
for p in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p],
                       capture_output=True, text=True)
    print(f'  {"✅" if r.returncode==0 else "⚠"} {p}')

import torch
print(f'\nPyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
print('✅ Cell 1 complete')

## Cell 2 — Upload Inputs
> Upload BOTH `batch_eval.zip` and `hdtf_prepared.zip`

In [ ]:
import os, zipfile, glob
from google.colab import files

os.makedirs('/content/metrics/gen', exist_ok=True)
os.makedirs('/content/metrics/real', exist_ok=True)

print('Upload batch_eval.zip AND hdtf_prepared.zip')
uploaded = files.upload()
for fname, data in uploaded.items():
    zpath = f'/content/{fname}'
    with open(zpath, 'wb') as f: f.write(data)
    if 'batch_eval' in fname.lower():
        with zipfile.ZipFile(zpath) as z: z.extractall('/content/metrics/gen')
        print(f'✅ {fname} → gen/')
    elif 'hdtf' in fname.lower() or 'prepared' in fname.lower():
        with zipfile.ZipFile(zpath) as z: z.extractall('/content/metrics/real')
        print(f'✅ {fname} → real/')

# Discover
def find_videos(pattern):
    out = {}
    for v in sorted(glob.glob(pattern, recursive=True)):
        stem = os.path.splitext(os.path.basename(v))[0]
        out[stem] = v
    return out

hybrid = find_videos('/content/metrics/gen/**/hybrid_lp/*.mp4')
lponly = find_videos('/content/metrics/gen/**/lp_only/*.mp4')
raw    = find_videos('/content/metrics/real/**/raw/*.mp4')

CONFIGS = {'hybrid_lp': hybrid, 'lp_only': lponly}
print(f'\n  hybrid_lp: {len(hybrid)} | lp_only: {len(lponly)} | raw(real): {len(raw)}')
common = sorted(set(hybrid) & set(lponly) & set(raw))
print(f'  Common identities: {len(common)} → {common}')

## Cell 3 — Frame Extraction Helpers

In [ ]:
import os, cv2, glob, numpy as np, subprocess, shlex

FRAME_BASE = '/content/metrics/frames'

def extract_frames(video, out_dir, max_frames=None, size=(299,299)):
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(video)
    i, saved = 0, 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        if size:
            frame = cv2.resize(frame, size)
        cv2.imwrite(f'{out_dir}/{saved:05d}.png', frame)
        saved += 1; i += 1
        if max_frames and saved >= max_frames: break
    cap.release()
    return saved

def read_frames_rgb(video, resize=None, max_frames=None):
    cap = cv2.VideoCapture(video)
    frames = []
    while True:
        ok, f = cap.read()
        if not ok: break
        if resize: f = cv2.resize(f, resize)
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        if max_frames and len(frames) >= max_frames: break
    cap.release()
    return np.array(frames)

print('✅ Helpers ready')

## Cell 4 — FID (Fréchet Inception Distance)

In [ ]:
import os, shutil, glob, torch
from pytorch_fid import fid_score

FRAME_BASE = '/content/metrics/frames'
if os.path.exists(FRAME_BASE): shutil.rmtree(FRAME_BASE)

# Build a 'real' frame pool from all raw videos
real_dir = f'{FRAME_BASE}/real'
print('Extracting real frames (raw HDTF)...')
for stem in common:
    extract_frames(raw[stem], real_dir + f'_{stem}', max_frames=60, size=(299,299))
# merge into one dir
os.makedirs(real_dir, exist_ok=True)
idx = 0
for d in glob.glob(f'{FRAME_BASE}/real_*'):
    for f in glob.glob(f'{d}/*.png'):
        shutil.move(f, f'{real_dir}/{idx:06d}.png'); idx += 1
    shutil.rmtree(d)
print(f'  Real frames: {idx}')

fid_results = {}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
for cfg, vids in CONFIGS.items():
    gen_dir = f'{FRAME_BASE}/{cfg}'
    print(f'\nExtracting {cfg} frames...')
    for stem in common:
        extract_frames(vids[stem], gen_dir + f'_{stem}', max_frames=60, size=(299,299))
    os.makedirs(gen_dir, exist_ok=True)
    idx = 0
    for d in glob.glob(f'{FRAME_BASE}/{cfg}_*'):
        if d == gen_dir: continue
        for f in glob.glob(f'{d}/*.png'):
            shutil.move(f, f'{gen_dir}/{idx:06d}.png'); idx += 1
        shutil.rmtree(d)
    print(f'  {cfg} frames: {idx}')

    print(f'  Computing FID for {cfg}...')
    fid = fid_score.calculate_fid_given_paths(
        [real_dir, gen_dir], batch_size=50, device=device, dims=2048)
    fid_results[cfg] = fid
    print(f'  FID({cfg}) = {fid:.3f}')

print('\n✅ FID done:', {k: round(v,3) for k,v in fid_results.items()})

## Cell 5 — FVD (Fréchet Video Distance)

In [ ]:
import numpy as np, torch
fvd_results = {}

try:
    from cdfvd import fvd as cdfvd_mod
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    def load_clips(video_dict, n_frames=16, size=(128,128)):
        clips = []
        for stem in common:
            fr = read_frames_rgb(video_dict[stem], resize=size, max_frames=n_frames)
            if len(fr) >= n_frames:
                clips.append(fr[:n_frames])
        return np.array(clips)  # (N, T, H, W, 3)

    real_clips = load_clips(raw)
    print(f'Real clips: {real_clips.shape}')

    evaluator = cdfvd_mod.cdfvd('i3d', device=device)
    evaluator.compute_real_stats(evaluator.load_videos(real_clips, data_type='numpy'))

    for cfg, vids in CONFIGS.items():
        gen_clips = load_clips(vids)
        print(f'{cfg} clips: {gen_clips.shape}')
        evaluator.compute_fake_stats(evaluator.load_videos(gen_clips, data_type='numpy'))
        score = evaluator.compute_fvd_from_stats()
        fvd_results[cfg] = float(score)
        print(f'  FVD({cfg}) = {score:.3f}')
        evaluator.empty_fake_stats()

    print('\n✅ FVD done:', {k: round(v,3) for k,v in fvd_results.items()})
except Exception as e:
    print(f'⚠ cd-fvd failed: {e}')
    print('  FVD will be marked N/A. (FID + temporal metrics still cover quality.)')
    fvd_results = {cfg: None for cfg in CONFIGS}

## Cell 6 — LPIPS / PSNR / SSIM (vs raw video, frame-aligned)

In [ ]:
import torch, lpips, numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
lpips_model = lpips.LPIPS(net='alex').to(device)

def align_and_score(gen_video, real_video, size=(256,256), max_frames=120):
    g = read_frames_rgb(gen_video, resize=size, max_frames=max_frames)
    r = read_frames_rgb(real_video, resize=size, max_frames=max_frames)
    n = min(len(g), len(r))
    if n == 0: return None, None, None
    g, r = g[:n], r[:n]

    # LPIPS (batched)
    g_t = torch.from_numpy(g).permute(0,3,1,2).float().to(device)/127.5 - 1
    r_t = torch.from_numpy(r).permute(0,3,1,2).float().to(device)/127.5 - 1
    lp_vals = []
    with torch.no_grad():
        for i in range(0, n, 16):
            d = lpips_model(g_t[i:i+16], r_t[i:i+16])
            lp_vals.extend(d.squeeze().cpu().numpy().flatten().tolist())
    lpips_mean = float(np.mean(lp_vals))

    # PSNR + SSIM (per frame)
    ps, ss = [], []
    for i in range(n):
        ps.append(psnr_fn(r[i], g[i], data_range=255))
        ss.append(ssim_fn(r[i], g[i], channel_axis=2, data_range=255))
    return lpips_mean, float(np.mean(ps)), float(np.mean(ss))

ref_results = {cfg: {} for cfg in CONFIGS}
for cfg, vids in CONFIGS.items():
    print(f'\n── {cfg} ──')
    for stem in common:
        lp, ps, ss = align_and_score(vids[stem], raw[stem])
        ref_results[cfg][stem] = {'lpips': lp, 'psnr': ps, 'ssim': ss}
        print(f'  {stem}: LPIPS={lp:.4f} PSNR={ps:.2f} SSIM={ss:.4f}')

print('\n✅ LPIPS/PSNR/SSIM done')

## Cell 7 — Aggregate & Export

In [ ]:
import numpy as np, json, csv
from datetime import datetime, timezone

def agg(cfg, metric):
    vals = [v[metric] for v in ref_results[cfg].values() if v.get(metric) is not None]
    a = np.array(vals, dtype=float)
    return (float(a.mean()), float(a.std())) if len(a) else (None, None)

print('='*80)
print('  CMPE492 — Extended Metrics (Multi-Identity, n={})'.format(len(common)))
print('='*80)
print(f'  {"Config":<12} {"FID↓":>10} {"FVD↓":>10} {"LPIPS↓":>14} {"PSNR↑":>14} {"SSIM↑":>14}')
print('-'*80)

summary = {}
for cfg in CONFIGS:
    lp_m, lp_s = agg(cfg, 'lpips')
    ps_m, ps_s = agg(cfg, 'psnr')
    ss_m, ss_s = agg(cfg, 'ssim')
    fid_v = fid_results.get(cfg)
    fvd_v = fvd_results.get(cfg)
    def f(v, d=3): return f'{v:.{d}f}' if v is not None else 'N/A'
    def fpm(m, s, d=3): return f'{m:.{d}f}±{s:.{d}f}' if m is not None else 'N/A'
    print(f'  {cfg:<12} {f(fid_v):>10} {f(fvd_v):>10} {fpm(lp_m,lp_s,4):>14} {fpm(ps_m,ps_s,2):>14} {fpm(ss_m,ss_s,4):>14}')
    summary[cfg] = {
        'fid': fid_v, 'fvd': fvd_v,
        'lpips_mean': lp_m, 'lpips_std': lp_s,
        'psnr_mean': ps_m, 'psnr_std': ps_s,
        'ssim_mean': ss_m, 'ssim_std': ss_s,
    }
print('='*80)

OUT = '/content/metrics/extended_metrics.json'
full = {'summary': summary, 'per_identity': ref_results,
        'fid': fid_results, 'fvd': fvd_results,
        'timestamp': datetime.now(timezone.utc).isoformat()}
with open(OUT, 'w') as f: json.dump(full, f, indent=2)
print(f'\n✅ Saved: {OUT}')

# CSV
csv_path = '/content/metrics/extended_per_identity.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['config','identity','lpips','psnr','ssim'])
    for cfg in CONFIGS:
        for stem, v in ref_results[cfg].items():
            w.writerow([cfg, stem, v['lpips'], v['psnr'], v['ssim']])
print(f'✅ Saved: {csv_path}')

from google.colab import files
files.download(OUT)
files.download(csv_path)
print('\n✅ Extended metrics complete')